# MCP server 6 — Vibration diagnostics

Run useful DSP-domain calculations without a database, then show where telemetry joins the flow.

**Tutorial contract:** run cells from top to bottom. Every external dependency is checked before use,
outputs go under `artifacts/kdd_tutorial/`, and no credential value is printed.


## Goal

Calculate bearing frequencies and ISO severity through MCP.

**Requires:** NumPy/SciPy; CouchDB only for real vibration telemetry


In [1]:
from pathlib import Path
import json, os, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


repo: /Users/chathurangishyalika/IBM/AssetOpsBench
python: 3.12.13


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()

if repo:
    load_dotenv(repo / ".env", override=False)

In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

ENTRY_POINTS = {
    "iot": "iot-mcp-server", "utilities": "utilities-mcp-server",
    "fmsr": "fmsr-mcp-server", "wo": "wo-mcp-server",
    "tsfm": "tsfm-mcp-server", "vibration": "vibration-mcp-server",
}

async def mcp_session(server, operation, tool_name=None, arguments=None):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), ENTRY_POINTS[server]],
        cwd=str(REPO),
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if operation == "list":
                return await session.list_tools()
            return await session.call_tool(tool_name, arguments or {})

def text_result(result):
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text

async def list_tools(server):
    response = await mcp_session(server, "list")
    return [{"name": t.name, "description": t.description, "schema": t.inputSchema} for t in response.tools]

async def call_tool(server, name, **arguments):
    return text_result(await mcp_session(server, "call", name, arguments))


## 1. Discover the live MCP contract

This starts the real stdio server and asks it for its tool schemas.


In [3]:
tools = await list_tools("vibration")
[(t["name"], list(t["schema"].get("properties", {}))) for t in tools]


[('get_vibration_data',
  ['site_name', 'asset_id', 'sensor_name', 'start', 'final']),
 ('list_vibration_sensors', ['site_name', 'asset_id']),
 ('compute_fft_spectrum', ['data_id', 'window', 'top_n']),
 ('compute_envelope_spectrum',
  ['data_id', 'band_low_hz', 'band_high_hz', 'top_n']),
 ('assess_vibration_severity', ['rms_velocity_mm_s', 'machine_group']),
 ('calculate_bearing_frequencies',
  ['rpm',
   'n_balls',
   'ball_diameter_mm',
   'pitch_diameter_mm',
   'contact_angle_deg',
   'bearing_name']),
 ('list_known_bearings', []),
 ('diagnose_vibration',
  ['data_id',
   'rpm',
   'bearing_designation',
   'bearing_n_balls',
   'bearing_ball_dia_mm',
   'bearing_pitch_dia_mm',
   'bearing_contact_angle_deg',
   'bpfo_hz',
   'bpfi_hz',
   'bsf_hz',
   'ftf_hz',
   'machine_group',
   'machine_description'])]

## 2. Credential-free bearing knowledge


In [4]:
bearings = await call_tool("vibration", "list_known_bearings")
bearings


{'bearings': [{'designation': '6205',
   'name': '6205 (Deep groove)',
   'n_balls': 9,
   'ball_dia_mm': 7.938,
   'pitch_dia_mm': 38.5,
   'contact_angle_deg': 0},
  {'designation': '6206',
   'name': '6206 (Deep groove)',
   'n_balls': 9,
   'ball_dia_mm': 9.525,
   'pitch_dia_mm': 46.0,
   'contact_angle_deg': 0},
  {'designation': '6207',
   'name': '6207 (Deep groove)',
   'n_balls': 9,
   'ball_dia_mm': 11.112,
   'pitch_dia_mm': 53.5,
   'contact_angle_deg': 0},
  {'designation': '6208',
   'name': '6208 (Deep groove)',
   'n_balls': 9,
   'ball_dia_mm': 12.7,
   'pitch_dia_mm': 60.0,
   'contact_angle_deg': 0},
  {'designation': '6305',
   'name': '6305 (Deep groove)',
   'n_balls': 8,
   'ball_dia_mm': 10.319,
   'pitch_dia_mm': 39.04,
   'contact_angle_deg': 0},
  {'designation': '6306',
   'name': '6306 (Deep groove)',
   'n_balls': 8,
   'ball_dia_mm': 12.303,
   'pitch_dia_mm': 46.36,
   'contact_angle_deg': 0},
  {'designation': 'NU205',
   'name': 'NU205 (Cylindrical ro

In [5]:
freqs = await call_tool("vibration", "calculate_bearing_frequencies", rpm=1800, n_balls=9, ball_diameter_mm=7.938, pitch_diameter_mm=38.5, contact_angle_deg=0, bearing_name="6205")
freqs


{'bearing': '6205',
 'rpm': 1800.0,
 'shaft_frequency_hz': 30.0,
 'ftf_hz': 11.907,
 'bpfo_hz': 107.165,
 'bpfi_hz': 162.835,
 'bsf_hz': 69.659,
 'harmonics': {'bpfo_2x': 214.331,
  'bpfo_3x': 321.496,
  'bpfi_2x': 325.669,
  'bpfi_3x': 488.504,
  'bsf_2x': 139.317}}

In [6]:
severity = await call_tool("vibration", "assess_vibration_severity", rms_velocity_mm_s=4.5, machine_group="group2")
severity


{'rms_velocity_mm_s': 4.5,
 'iso_zone': 'C',
 'description': 'Alarm - not suitable for long-term operation',
 'machine_group': 'group2',
 'thresholds': {'A_good': 1.4, 'B_acceptable': 2.8, 'C_alarm': 7.1}}

## 3. Optional data-backed path

With seeded CouchDB, call `get_vibration_data` first; use its returned `data_id` for FFT, envelope, and diagnosis.


In [7]:
import socket
from urllib.parse import urlparse

def tcp_reachable(url, timeout=1.0):
    parsed = urlparse(url)
    host, port = parsed.hostname or "localhost", parsed.port or 5984
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

COUCHDB_URL = os.getenv("COUCHDB_URL", "http://localhost:5984")
print("CouchDB reachable:", tcp_reachable(COUCHDB_URL), "at", COUCHDB_URL)


CouchDB reachable: True at http://localhost:5984


## Takeaway

You exercised the server through MCP JSON-RPC over stdio—the same boundary the agents use.
